# Add standard sequence keys to an SWN vec

The SWN (Shifting White Noise) stimulus is built exactly like the checkerboard: every
sequence is **1200 frames = 600 novel frames** (never repeated) **followed by 600 repeated
frames** that are identical in every sequence. With 54000 triggers that is **45 repetitions
of a 600-frame (20 s at 30 Hz) repeated block**.

The raw SWN vec leaves its last column at 0, so no analysis can find those repetitions.
This notebook fills that last column with the standard sequence keys, exactly like the
chirp and drifting-gratings vecs:

    key = sequence_number * 10000 + repetition        (the last 4 digits are the repetition)

The repeated block becomes sequence **1**, with repetitions 0-44 (keys 10000 ... 10044).
Novel frames keep the key 0 (they are not a repeated sequence).

The resulting `*_std.vec` can then be used by the standard vec analysis
(`6_Standard_Vec_Analysis.ipynb`) like any other stimulus.

In [ ]:
import numpy as np

vec_file = "20250512_4_SWN_48pixCh_6pixShift_30Hz_MEA2.vec"

# Load vec file
vec = np.loadtxt(vec_file)[1:, :]  # Remove first line as it is not a trigger
vec_header = np.loadtxt(vec_file, max_rows=1)
print(f"\nSelected vec file : {vec_file}\n")
print(f"Vec file length : {vec.shape[0]}")

## Check the structure

The frame-index column (column 1) points into the .bin. The novel frames use the low
indices, the repeated block always re-uses the same high indices — so we can see the
novel / repeated alternation directly.

In [ ]:
FRAMES_PER_SEQUENCE = (
    1200  # one SWN sequence: 600 novel frames, then 600 repeated frames
)

frame_idx = vec[:, 1].astype(int)
n_reps = len(vec) // FRAMES_PER_SEQUENCE
sequences = frame_idx.reshape(n_reps, FRAMES_PER_SEQUENCE)
half = FRAMES_PER_SEQUENCE // 2

# The repeated frames are the ones re-used by every sequence.
repeated_frames = np.unique(sequences[:, half:])
print(f"{n_reps} sequences of {FRAMES_PER_SEQUENCE} frames")
print(
    f"repeated block: {len(repeated_frames)} distinct frames "
    f"(bin indices {repeated_frames.min()}-{repeated_frames.max()})"
)
print(
    f"second half identical in every sequence : {np.all(sequences[:, half:] == sequences[0, half:])}"
)
print(
    f"first half never re-used (novel)        : {len(np.unique(sequences[:, :half])) == n_reps * half}"
)

## Write the standard keys

In [ ]:
SEQUENCE_KEY = 1  # the repeated block is sequence "1"
REP_MAX_LENGTH = 1000  # key = SEQUENCE_KEY * REP_MAX_LENGTH * 10 + rep -> 4 digits for the repetition

vec_std = vec.copy()
for rep in range(n_reps):
    start = rep * FRAMES_PER_SEQUENCE + half  # the repeated half of this sequence
    vec_std[start : start + half, -1] = SEQUENCE_KEY * REP_MAX_LENGTH * 10 + rep

keys = vec_std[:, -1].astype(int)
print(f"keys written : {keys[keys > 0].min()} ... {keys[keys > 0].max()}")
print(f"rows per repetition : {np.bincount(keys[keys > 0] - 10000)[0]}")
print(f"rows left at key 0 (novel frames) : {(keys == 0).sum()}")

# sanity check: the rows we keyed must be exactly the rows that use a repeated frame
assert np.array_equal(keys > 0, frame_idx >= repeated_frames.min())
print("\nOK: keyed rows are exactly the rows showing a repeated frame.")

In [ ]:
full_std_vec = np.concatenate((vec_header.reshape(1, -1), vec_std), axis=0)

np.savetxt(
    "20250512_4_SWN_48pixCh_6pixShift_30Hz_MEA2_std.vec", full_std_vec, fmt="%1.f"
)
print("saved 20250512_4_SWN_48pixCh_6pixShift_30Hz_MEA2_std.vec")